# BRSET exploratory follow-ups (after plan v6)

Follows `04_results/brset_exploratory_plan.json`. **Exploratory; changes no confirmatory result.**
- **Part A:** does the training-data result (B1) survive model strength? DDR-trained models are weaker on BRSET.
- **Part B:** why did the sibling rule (B4) miss its bar? Separation (AUC) and flag rates with agreement measured at wider referral shares and by score ranking.

Aggregate output only: `04_results/brset_exploratory_results.json`. CPU, High-RAM; about 15 to 30 minutes.

In [1]:
# Cell 1: guards and loading
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, hashlib, itertools, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr, norm, multivariate_normal
from scipy.optimize import brentq
from sklearn.metrics import roc_auc_score
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'; BR = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'
PREDS, RESULTS, AUDIT = f'{BASE}/03_predictions', f'{BASE}/04_results', f'{BASE}/99_audit'
OUT = f'{RESULTS}/brset_exploratory_results.json'
assert os.path.exists(f'{RESULTS}/brset_results_v6.json'), 'Run the confirmatory analysis first.'
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
LOCK = f'{RESULTS}/locked_config_v6_brset.json'
IREC = json.load(open(f'{AUDIT}/brset_inference_record_v2.json')); ACC = json.load(open(f'{BR}/brset_access_record.json'))
assert sha(LOCK) == IREC['lock_v6_sha256'], 'Plan changed after inference.'
assert sha(f'{BR}/label_brset.csv') == ACC['label_file_sha256'] == IREC['access_record_label_sha256'], 'Label file changed.'
for fn, h in IREC['prediction_sha256'].items(): assert sha(f'{PREDS}/{fn}') == h, f'prediction file changed: {fn}'
print('plan, label file and 45 prediction files match their fingerprints')
L = pd.read_csv(f'{BR}/label_brset.csv', dtype=str)
cols = list(L.columns); low = {c: c.lower() for c in cols}
icdr = [c for c in cols if 'icdr' in low[c]]; pid = [c for c in cols if low[c] in ('patient_id', 'patient')]; iid = [c for c in cols if low[c] in ('image_id', 'image')]
qual = [c for c in cols if low[c] == 'quality']
if len(icdr) != 1 or len(pid) != 1 or len(iid) != 1:
    raise SystemExit(f'Plan rule: ambiguous columns, stop for a recorded amendment. Column names: {cols}')
ICDR, PID, IID = icdr[0], pid[0], iid[0]
g = pd.to_numeric(L[ICDR], errors='coerce')
coh = L[g.isin([0, 1, 2, 3, 4])].copy(); coh['grade'] = g[g.isin([0, 1, 2, 3, 4])].astype(int)
coh['image_id'] = coh[IID].astype(str).str.replace(r'\.(jpg|jpeg|png)$', '', regex=True)
SRC = [('eyepacs', m) for m in ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
       'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']] + [('ddr', m) for m in ['densenet121_5class','efficientnet_b3_5class','resnet50_5class','retfound_5class']]
def tag(k): return '' if k == 0 else f'_seed{k}'
P = {}
for s, m in SRC:
    for k in (0, 1, 2):
        d = pd.read_csv(f'{PREDS}/brset_predictions_{s}_{m}{tag(k)}_v2.csv', dtype={'image_id': str}).set_index('image_id').predicted_probability
        P[(s, m, k)] = d
common = sorted(set(coh.image_id) & set(next(iter(P.values())).index)); coh = coh.set_index('image_id').loc[common].reset_index()
P = {key: v.reindex(coh.image_id).to_numpy() for key, v in P.items()}
assert all(np.isfinite(v).all() for v in P.values())
G = coh.grade.to_numpy(); CL = pd.factorize(coh[PID].astype(str))[0]
counts = {'images_in_label_file': int(len(L)), 'cohort_images': int(len(coh)), 'patients': int(CL.max() + 1),
          'grade_counts': {int(k): int(v) for k, v in pd.Series(G).value_counts().sort_index().items()},
          'excluded_no_valid_grade': int(len(L) - (g.isin([0, 1, 2, 3, 4])).sum()), 'icdr_column': ICDR, 'quality_column': qual[0] if qual else None}
print('cohort', len(coh), 'images')

Mounted at /content/drive
plan, label file and 45 prediction files match their fingerprints
cohort 16266 images


In [2]:
# Cell 2: helpers (referral, bootstrap, combos)
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
RNG = np.random.default_rng(20261003); NB = 1000
def ci(v): v = np.asarray(v, float); v = v[np.isfinite(v)]; return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)] if len(v) else [None, None]
def combos(va, vb, seeds=(0, 1, 2)):
    if va == vb: return [((*va, i), (*vb, j)) for i in seeds for j in seeds if i < j]
    return [((*va, i), (*vb, j)) for i in seeds for j in seeds]
def setup(mask):
    S = {'G': G[mask], 'CL': pd.factorize(CL[mask])[0]}
    S['ref'] = S['G'] >= 2; S['sev'] = S['G'] >= 3; S['pdr'] = S['G'] == 4
    S['f'] = 1.31 * S['ref'].mean(); S['f2'] = S['ref'].mean(); n = int(mask.sum())
    S['P'] = {k: v[mask] for k, v in P.items()}; S['RK'] = {k: rankdata(v) / n for k, v in S['P'].items()}
    ncl = S['CL'].max() + 1; S['W'] = [np.bincount(RNG.integers(0, ncl, ncl), minlength=ncl)[S['CL']].astype(float) for _ in range(NB)]
    return S
ALL = setup(np.ones(len(G), bool))
CORE_V = [(s, a) for s in ('eyepacs', 'ddr') for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class')]
EYE_V = [v for v in SRC if v[0] == 'eyepacs']
AUC = {f'{s}_{m}{tag(k)}': round(float(roc_auc_score(ALL['ref'], ALL['P'][(s, m, k)])), 4) for s, m in SRC for k in (0, 1, 2)}
print('cohort f =', round(ALL['f'], 4), '| referable', int(ALL['ref'].sum()), '| severe', int(ALL['sev'].sum()), '| PDR', int(ALL['pdr'].sum()))

cohort f = 0.0745 | referable 925 | severe 474 | PDR 396


In [3]:
# Cell 3: Part A, strength check on B1
def ptype4(va, vb):
    if va == vb: return 'repeat_runs'
    if va[1] == vb[1]: return 'same_design_diff_source'
    if va[0] != vb[0]: return 'diff_design_diff_source'
    return 'diff_design_same_source_' + va[0]
def partA(S, tname):
    tgt = S[tname]; R = {k: refer(v, S['f']) for k, v in S['P'].items() if k[:2] in CORE_V}; VEC = {}
    for i, va in enumerate(CORE_V):
        for vb in CORE_V[i + 1:]:
            if va[1] == vb[1]: continue
            cs = combos(va, vb); ma = mb = both = 0
            for ka, kb in cs:
                x, y = tgt & ~R[ka], tgt & ~R[kb]; ma = ma + x; mb = mb + y; both = both + (x & y)
            VEC[(va, vb)] = (ma / len(cs), mb / len(cs), both / len(cs))
    groups = ['diff_design_same_source_eyepacs', 'diff_design_same_source_ddr', 'diff_design_diff_source']
    def stats(w):
        T = tgt @ w; out = {g_: [] for g_ in groups}; rows = []
        for p, (ma, mb, both) in VEC.items():
            a_, b_, c_ = ma @ w, mb @ w, both @ w
            if a_ * b_ <= 0: continue
            r = c_ / (a_ * b_ / T); out[ptype4(*p)].append(r)
            rows.append((r, float(p[0][0] == p[1][0]), (a_ + b_) / 2, abs(a_ - b_)))
        m = {g_: float(np.mean(v)) for g_, v in out.items()}
        X = np.array(rows); Z = np.column_stack([np.ones(len(X)), X[:, 1], X[:, 2] / X[:, 2].mean(), X[:, 3] / max(X[:, 3].mean(), 1e-9)])
        beta = np.linalg.lstsq(Z, X[:, 0], rcond=None)[0]
        m['eyepacs_pairs_minus_cross'] = m['diff_design_same_source_eyepacs'] - m['diff_design_diff_source']
        m['ddr_pairs_minus_cross'] = m['diff_design_same_source_ddr'] - m['diff_design_diff_source']
        m['same_source_adjusted_for_miss_counts'] = float(beta[1]); return m
    obs = stats(np.ones(len(tgt))); bs = [stats(w) for w in S['W'][:500]]
    return {k: {'mean': round(v, 3), 'ci95': ci([b[k] for b in bs])} for k, v in obs.items()}
PA = {'referable': partA(ALL, 'ref'), 'severe': partA(ALL, 'sev')}
print(json.dumps(PA, indent=1))

{
 "referable": {
  "diff_design_same_source_eyepacs": {
   "mean": 2.694,
   "ci95": [
    2.4831,
    2.921
   ]
  },
  "diff_design_same_source_ddr": {
   "mean": 1.441,
   "ci95": [
    1.3864,
    1.5041
   ]
  },
  "diff_design_diff_source": {
   "mean": 1.682,
   "ci95": [
    1.6004,
    1.7707
   ]
  },
  "eyepacs_pairs_minus_cross": {
   "mean": 1.012,
   "ci95": [
    0.855,
    1.1927
   ]
  },
  "ddr_pairs_minus_cross": {
   "mean": -0.24,
   "ci95": [
    -0.286,
    -0.2019
   ]
  },
  "same_source_adjusted_for_miss_counts": {
   "mean": 0.256,
   "ci95": [
    0.1875,
    0.3445
   ]
  }
 },
 "severe": {
  "diff_design_same_source_eyepacs": {
   "mean": 4.354,
   "ci95": [
    3.6347,
    5.2383
   ]
  },
  "diff_design_same_source_ddr": {
   "mean": 1.592,
   "ci95": [
    1.491,
    1.7182
   ]
  },
  "diff_design_diff_source": {
   "mean": 1.937,
   "ci95": [
    1.7518,
    2.1141
   ]
  },
  "eyepacs_pairs_minus_cross": {
   "mean": 2.417,
   "ci95": [
    1.7964,


In [4]:
# Cell 4: Part B, sibling diagnostics (agreement needs no labels)
CORE = [(*v, k) for v in CORE_V for k in (0, 1, 2)]
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
def ptype(va, vb):
    if va == vb: return 'repeat_runs'
    if va[1] == vb[1]: return 'same_design_diff_source'
    return 'diff_design'
PAIRS = list(itertools.combinations(CORE, 2))
def measure(kind, share=None):
    if kind == 'kappa': R = {k: refer(ALL['P'][k], share) for k in CORE}; return [kappa(R[a], R[b]) for a, b in PAIRS]
    return [float(np.corrcoef(ALL['RK'][a], ALL['RK'][b])[0, 1]) for a, b in PAIRS]
TY = np.array([ptype(a[:2], b[:2]) for a, b in PAIRS]); SRCP = np.array([a[0] if a[:2] == b[:2] else '' for a, b in PAIRS])
PB = {}
for label, kind, share in [(f'kappa_at_plan_share_{ALL["f"]:.3f}', 'kappa', ALL['f']), ('kappa_at_0.15', 'kappa', 0.15), ('kappa_at_0.25', 'kappa', 0.25),
                           ('kappa_at_0.50', 'kappa', 0.50), ('rank_correlation_of_scores', 'rank', None)]:
    v = np.array(measure(kind, share)); sib = TY == 'repeat_runs'; dd = TY == 'diff_design'
    thr = float(np.percentile(v[dd], 95))
    PB[label] = {'auc_repeat_vs_diff_design': round(float(roc_auc_score(np.r_[np.ones(sib.sum()), np.zeros(dd.sum())], np.r_[v[sib], v[dd]])), 3),
                 'relative_rule_flag_rate_repeat_runs': round(float((v[sib] > thr).mean()), 3),
                 'flag_rate_eyepacs_repeat_runs': round(float((v[sib & (SRCP == 'eyepacs')] > thr).mean()), 3),
                 'flag_rate_ddr_repeat_runs': round(float((v[sib & (SRCP == 'ddr')] > thr).mean()), 3),
                 'flag_rate_same_design_diff_source': round(float((v[TY == 'same_design_diff_source'] > thr).mean()), 3),
                 'median_by_type': {t: round(float(np.median(v[TY == t])), 3) for t in ('repeat_runs', 'same_design_diff_source', 'diff_design')}}
print(json.dumps(PB, indent=1))
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'brset_exploratory_plan.json', 'status': 'exploratory',
       'model_auc_core24': {k: v for k, v in AUC.items() if any(a in k for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class'))},
       'partA_strength': PA, 'partB_siblings': PB}
json.dump(res, open(OUT, 'w'), indent=2, default=str)
drive.flush_and_unmount(); drive.mount('/content/drive')
print('Wrote 04_results/brset_exploratory_results.json')

{
 "kappa_at_plan_share_0.074": {
  "auc_repeat_vs_diff_design": 0.879,
  "relative_rule_flag_rate_repeat_runs": 0.625,
  "flag_rate_eyepacs_repeat_runs": 1.0,
  "flag_rate_ddr_repeat_runs": 0.25,
  "flag_rate_same_design_diff_source": 0.083,
  "median_by_type": {
   "repeat_runs": 0.728,
   "same_design_diff_source": 0.499,
   "diff_design": 0.456
  }
 },
 "kappa_at_0.15": {
  "auc_repeat_vs_diff_design": 0.952,
  "relative_rule_flag_rate_repeat_runs": 0.75,
  "flag_rate_eyepacs_repeat_runs": 1.0,
  "flag_rate_ddr_repeat_runs": 0.5,
  "flag_rate_same_design_diff_source": 0.25,
  "median_by_type": {
   "repeat_runs": 0.659,
   "same_design_diff_source": 0.451,
   "diff_design": 0.415
  }
 },
 "kappa_at_0.25": {
  "auc_repeat_vs_diff_design": 0.988,
  "relative_rule_flag_rate_repeat_runs": 0.875,
  "flag_rate_eyepacs_repeat_runs": 1.0,
  "flag_rate_ddr_repeat_runs": 0.75,
  "flag_rate_same_design_diff_source": 0.306,
  "median_by_type": {
   "repeat_runs": 0.634,
   "same_design_diff_so